# UDFs vs Vectorized Pandas UDFs: Architecture & Performance Benchmarks

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 08_udfs_and_pandas_udfs.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How standard Python UDFs work under Py4J, how Apache Arrow enables Vectorized Pandas UDFs, and measuring performance differences.
- **Why does this topic exist?**: Standard Python UDFs are one of the biggest performance killers in Spark. Vectorized UDFs provide 10x-50x speedups.
- **What problem does it solve?**: Row-by-row socket serialization between JVM and Python worker processes creates massive CPU and GC overhead.
- **Where is it used in real-world Data Engineering?**: Running machine learning model inference and complex domain logic in PySpark.

---

## 2. Python UDF vs Pandas UDF Execution Models

```
Standard Python UDF (Row-by-Row Serialization):
[JVM Executor Heap] -> [Socket / Pickle] -> [Python Daemon] -> [Execute Function]
         ^                                                               |
         +------------------- [Socket / Unpickle] <----------------------+
(Severe overhead: 1 serialization event PER ROW; No Catalyst optimization)

Vectorized Pandas UDF (Apache Arrow Shared Memory):
[JVM Executor Heap] ======= Apache Arrow RecordBatch =======> [Python Worker]
                              (Zero-Copy / Bulk SIMD)         [Pandas Series vectorized]
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("UDF_Benchmarking_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Implementing Standard Python UDF vs Pandas UDF

In [ ]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

# 1. Standard Python UDF
@udf(returnType=DoubleType())
def standard_tax_udf(price, quantity):
    if price is None or quantity is None:
        return 0.0
    return float(price * quantity * 0.14)

# 2. Vectorized Pandas UDF (Series to Series)
@pandas_udf(DoubleType())
def vectorized_tax_udf(price: pd.Series, quantity: pd.Series) -> pd.Series:
    return price.fillna(0.0) * quantity.fillna(0) * 0.14

# 3. Native Built-in Function (Baseline)
native_expr = F.round(F.coalesce(F.col("unit_price"), F.lit(0.0)) * F.coalesce(F.col("quantity"), F.lit(0)) * 0.14, 2)

## 4. Benchmark: Native vs Pandas UDF vs Standard Python UDF

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# Duplicate dataset to 240,000 rows to create meaningful timing differences
bench_df = orders.union(orders).union(orders).union(orders).cache()
bench_df.count() # Force cache population

# Benchmark 1: Native Spark Function
t0 = time.time()
bench_df.withColumn("tax", native_expr).write.mode("overwrite").format("noop").save()
t_native = time.time() - t0

# Benchmark 2: Vectorized Pandas UDF (Arrow)
t0 = time.time()
bench_df.withColumn("tax", vectorized_tax_udf(F.col("unit_price"), F.col("quantity"))).write.mode("overwrite").format("noop").save()
t_pandas = time.time() - t0

# Benchmark 3: Standard Python UDF
t0 = time.time()
bench_df.withColumn("tax", standard_tax_udf(F.col("unit_price"), F.col("quantity"))).write.mode("overwrite").format("noop").save()
t_standard = time.time() - t0

print(f"=== Execution Time Benchmark (240k rows) ===")
print(f"1. Native Spark Function: {t_native:.3f}s (Fastest - runs pure JVM)")
print(f"2. Vectorized Pandas UDF: {t_pandas:.3f}s (Arrow Batch Vectorization)")
print(f"3. Standard Python UDF:   {t_standard:.3f}s (Slowest - row-by-row socket pickle)")
print(f"Speedup of Vectorized over Standard: {t_standard / t_pandas:.2f}x")
bench_df.unpersist()

---

## Experiment: Inspecting Whole-Stage Codegen Suppression

- **1. What are we testing?**: Show that standard Python UDF suppresses Whole-Stage Code Generation while Native operations do not.
- **2. What do we expect?**: Native expression shows WholeStageCodegen (*) in explain(); standard UDF drops out of codegen.

In [ ]:
print("=== Native Expression Plan (WholeStageCodegen active) ===")
orders.select(native_expr).explain()

print("\n=== Standard Python UDF Plan (BatchEvalPython operator) ===")
orders.select(standard_tax_udf(F.col("unit_price"), F.col("quantity"))).explain()

**3. What happened?**: The standard UDF introduced BatchEvalPython and disabled WholeStageCodegen.  
**4. Why did it happen?**: Spark must serialize rows out of the JVM into external Python worker processes.  
**5. Practical Engineering Lesson**: Always exhaust native pyspark.sql.functions before resorting to custom Python functions.

---

## Practical Exercises

### Exercise 1 (Beginner): String Cleansing Pandas UDF
**Problem Statement**: Write a Pandas UDF that takes a Series of strings and strips whitespace and lowercases all characters.  
**Input / Context**: Series of channel strings

In [ ]:
# Exercise 1: Clean string Pandas UDF

### Exercise 2 (Intermediate): Grouped Map Pandas UDF with applyInPandas
**Problem Statement**: Use applyInPandas() to subtract the channel mean unit_price from each individual order within each channel.  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 2: applyInPandas

### Exercise 3 (Advanced): Scalar Pandas UDF with Multiple Inputs
**Problem Statement**: Implement a Haversine distance or compound interest formula using vectorized NumPy inside a Pandas UDF.  
**Input / Context**: Two numeric columns

In [ ]:
# Exercise 3: Vectorized math

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (String Cleansing Pandas UDF):
```python
@pandas_udf(StringType())
def clean_str_udf(s: pd.Series) -> pd.Series:
    return s.str.strip().str.lower()

orders.select(clean_str_udf(F.col("channel")).alias("clean_channel")).show(3)
```

#### Solution 2 (Grouped Map Pandas UDF with applyInPandas):
```python
def subtract_mean(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf['price_diff'] = pdf['unit_price'] - pdf['unit_price'].mean()
    return pdf[['order_id','channel','unit_price','price_diff']]

res = orders.groupBy("channel").applyInPandas(subtract_mean, schema="order_id int, channel string, unit_price double, price_diff double")
res.show(5)
```

#### Solution 3 (Scalar Pandas UDF with Multiple Inputs):
```python
import numpy as np

@pandas_udf(DoubleType())
def compound_growth(principal: pd.Series, rate: pd.Series) -> pd.Series:
    return principal * np.power(1.0 + rate, 3)

spark.range(3).withColumn("growth", compound_growth(F.lit(100.0), F.lit(0.05))).show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Hierarchy of Performance**: 1. Native Functions (Best) > 2. Pandas UDF (Good) > 3. Standard UDF (Avoid).
- **Arrow Zero-Copy**: Pandas UDFs transfer entire batches using Apache Arrow memory format.
- **Whole-Stage Codegen**: Python UDFs break operator fusion across the stage.

### Essential Technical Terms
- `Pandas UDF`: Vectorized UDF using Apache Arrow and Pandas Series.
- `BatchEvalPython`: Physical operator indicating Python socket execution.
- `Apache Arrow`: In-memory columnar data format enabling zero-copy cross-language sharing.

### Common Engineering Mistakes
- **Writing UDFs for existing functions**: e.g. writing a UDF for lower() or round() kills performance.
- **Returning scalar instead of Series**: A Series-to-Series Pandas UDF must accept and return pd.Series of identical length.

### Interview Questions on This Topic
- **Q: Why are standard Python UDFs slow in PySpark?**
  - *A*: They require row-by-row serialization across a socket between the JVM and Python worker, disabling Catalyst optimizations and Whole-Stage Codegen.
- **Q: How do Pandas UDFs achieve 10x-50x speedups?**
  - *A*: They use Apache Arrow to transfer entire columnar record batches directly into memory without row-level serialization overhead.

### What to Study Next
Module 03 complete! Proceed to Module 04: Spark Execution Engine Deep Dive.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')